# 02 — Hypergraph Construction
**Reddit Opinion & Community Research — Pipeline step 2 of 11**


In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg in ["pandas", "pyarrow", "numpy"]:
    ensure(pkg)

print("Packages OK.")


Packages OK.


In [2]:
import os, json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

_default_base = "/content/drive/MyDrive/PhD/reddit_opinion_research" if IN_COLAB else os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")
CONTRACT_PATH = os.path.join(_default_base, "processed_v2", "raw_enrichment_contract.json")

with open(CONTRACT_PATH, "r", encoding="utf-8") as f:
    contract = json.load(f)

PROCESSED_V2_DIR = contract["processed_v2_dir"]
ENGINEERED_DIR = os.path.join(PROCESSED_V2_DIR, "engineered")
os.makedirs(ENGINEERED_DIR, exist_ok=True)

print("Loaded contract from:", CONTRACT_PATH)
print("PROCESSED_V2_DIR:", PROCESSED_V2_DIR)
print("ENGINEERED_DIR:  ", ENGINEERED_DIR)


Mounted at /content/drive
Loaded contract from: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/raw_enrichment_contract.json
PROCESSED_V2_DIR: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2
ENGINEERED_DIR:   /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/engineered


In [3]:
text_corpus = pd.read_parquet(contract["paths"]["text_corpus"])
temporal_edges = pd.read_parquet(contract["paths"]["temporal_edges"])
static_edges = pd.read_parquet(contract["paths"]["static_edges"])
coparticipation_edges = pd.read_parquet(contract["paths"]["coparticipation_edges"])
node_features = pd.read_parquet(contract["paths"]["node_features"])

for name, df in [("text_corpus", text_corpus), ("temporal_edges", temporal_edges),
                  ("static_edges", static_edges), ("coparticipation_edges", coparticipation_edges),
                  ("node_features", node_features)]:
    print(f"{name}: {df.shape}")


text_corpus: (210994, 12)
temporal_edges: (194494, 7)
static_edges: (157953, 6)
coparticipation_edges: (1891957, 3)
node_features: (22465, 8)


In [4]:
text_corpus["created_ts"] = pd.to_datetime(text_corpus["created_utc"], unit="s", utc=True)


## Coverage check 

In [5]:
authors_in_corpus = set(text_corpus["author_hash"].dropna().unique())
authors_in_nodes = set(node_features["author_hash"].dropna().unique())
missing = authors_in_corpus - authors_in_nodes

print(f"Unique authors in text_corpus: {len(authors_in_corpus):,}")
print(f"Unique authors in node_features: {len(authors_in_nodes):,}")
print(f"Corpus authors missing node features: {len(missing):,} "
      f"({len(missing)/len(authors_in_corpus)*100:.2f}%)")
if len(missing) == 0:
    print("-> Full coverage, as expected from building node_features directly off text_corpus.")


Unique authors in text_corpus: 22,465
Unique authors in node_features: 22,465
Corpus authors missing node features: 0 (0.00%)
-> Full coverage, as expected from building node_features directly off text_corpus.


## Thread-size / hyperedge cardinality 

In [6]:
docs_per_thread = text_corpus.groupby("thread_id").size()
authors_per_thread = text_corpus.groupby("thread_id")["author_hash"].nunique()
duration = text_corpus.groupby("thread_id")["created_ts"].agg(lambda s: (s.max() - s.min()).total_seconds())

print(f"Number of threads (hyperedges): {docs_per_thread.shape[0]:,}\n")
print("Documents per thread:")
print(docs_per_thread.describe())
print(docs_per_thread.quantile([0.50, 0.90, 0.95, 0.99]))

print("\nDistinct authors per thread:")
print(authors_per_thread.describe())
print(authors_per_thread.quantile([0.50, 0.90, 0.95, 0.99]))

print("\nThread duration (hours):")
print((duration / 3600).describe())
print((duration / 3600).quantile([0.50, 0.90, 0.95, 0.99]))


Number of threads (hyperedges): 12,932

Documents per thread:
count    12932.000000
mean        16.315651
std         32.989048
min          1.000000
25%          1.000000
50%          5.000000
75%         16.000000
max        676.000000
dtype: float64
0.50      5.00
0.90     44.00
0.95     70.00
0.99    157.69
dtype: float64

Distinct authors per thread:
count    12932.000000
mean         9.230668
std         16.465286
min          1.000000
25%          1.000000
50%          3.000000
75%         10.000000
max        293.000000
Name: author_hash, dtype: float64
0.50     3.00
0.90    24.00
0.95    38.00
0.99    79.69
Name: author_hash, dtype: float64

Thread duration (hours):
count    12932.000000
mean       132.437481
std        472.977330
min          0.000000
25%          0.000000
50%          4.379167
75%         42.808194
max       4267.789444
Name: created_ts, dtype: float64
0.50       4.379167
0.90     184.407389
0.95     665.329222
0.99    2862.033328
Name: created_ts, dtype: fl

## Relation-type breakdown across the three relations

In [7]:
print("temporal_edges edge_type counts:")
print(temporal_edges["edge_type"].value_counts())
print(f"\ncoparticipation_edges: {len(coparticipation_edges):,} rows")

threads_with_reply = set(temporal_edges["thread_id"].unique())
threads_with_coparticipation = set(coparticipation_edges["thread_id"].unique())
all_threads = set(text_corpus["thread_id"].unique())

only_coparticipation = threads_with_coparticipation - threads_with_reply
only_reply = threads_with_reply - threads_with_coparticipation
neither = all_threads - threads_with_reply - threads_with_coparticipation

print(f"\nThreads with at least one reply edge: {len(threads_with_reply):,}")
print(f"Threads with at least one co-participation edge: {len(threads_with_coparticipation):,}")
print(f"Threads with co-participation but no direct reply edge: {len(only_coparticipation):,}")
print(f"Threads with neither relation (single-document threads): {len(neither):,}")


temporal_edges edge_type counts:
edge_type
reply_to_comment    110720
reply_to_post        83774
Name: count, dtype: int64

coparticipation_edges: 1,891,957 rows

Threads with at least one reply edge: 8,887
Threads with at least one co-participation edge: 9,183
Threads with co-participation but no direct reply edge: 313
Threads with neither relation (single-document threads): 3,732


## Chronological order vs. reply-tree order 



In [8]:
def analyze_thread(g):
    g_sorted = g.sort_values("created_ts")
    doc_ids = list(g_sorted["doc_id"])
    parents = list(g_sorted["parent_doc_id"])

    chain_matches = 0
    checked = 0
    for i in range(1, len(doc_ids)):
        if parents[i] is not None:
            checked += 1
            if parents[i] == doc_ids[i - 1]:
                chain_matches += 1

    return pd.Series({
        "n_docs": len(g_sorted),
        "n_unique_authors": g_sorted["author_hash"].nunique(),
        "start_time_unix": int(g_sorted["created_utc"].min()),
        "end_time_unix": int(g_sorted["created_utc"].max()),
        "duration_seconds": int(g_sorted["created_utc"].max() - g_sorted["created_utc"].min()),
        "doc_ids": doc_ids,
        "author_hashes": list(g_sorted["author_hash"]),
        "timestamps_unix": [int(x) for x in g_sorted["created_utc"]],
        "parent_doc_ids": parents,
        "reply_edge_types": list(g_sorted["reply_edge_type"]),
        "sentiment_labels": [None if pd.isna(x) else str(x) for x in g_sorted["sentiment_label"]],
        "topic_ids": [None if pd.isna(x) else str(x) for x in g_sorted["topic_id"]],
        "chain_order_matches": chain_matches,
        "chain_order_checked": checked,
    })

hyperedges = (
    text_corpus.groupby("thread_id", sort=False)
    .apply(analyze_thread, include_groups=False)
    .reset_index()
)

total_matches = hyperedges["chain_order_matches"].sum()
total_checked = hyperedges["chain_order_checked"].sum()
pct = total_matches / total_checked * 100 if total_checked else float("nan")
print(f"Chronological order matches direct-parent order for {total_matches:,}/{total_checked:,} replies ({pct:.1f}%)")
print("-> Above ~90%: chronological order is a fine default. Meaningfully lower: worth switching HOD-Mamba's")
print("   fine-grained scan to reply-tree order (a tree traversal using parent_doc_id) instead.")


Chronological order matches direct-parent order for 40,876/198,062 replies (20.6%)
-> Above ~90%: chronological order is a fine default. Meaningfully lower: worth switching HOD-Mamba's
   fine-grained scan to reply-tree order (a tree traversal using parent_doc_id) instead.


In [9]:
hyperedges_path = os.path.join(ENGINEERED_DIR, "hyperedges.parquet")
hyperedges.to_parquet(hyperedges_path, engine="pyarrow")
print("Saved:", hyperedges_path)


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/engineered/hyperedges.parquet


## Write the hypergraph contract 

In [10]:
hypergraph_contract = {
    "generated_from": CONTRACT_PATH,
    "engineered_dir": ENGINEERED_DIR,
    "paths": {
        "hyperedges": hyperedges_path,
        "node_features": contract["paths"]["node_features"],
        "temporal_edges": contract["paths"]["temporal_edges"],
        "static_edges": contract["paths"]["static_edges"],
        "coparticipation_edges": contract["paths"]["coparticipation_edges"],
    },
    "stats": {
        "n_hyperedges": int(docs_per_thread.shape[0]),
        "docs_per_thread_percentiles": {str(k): float(v) for k, v in docs_per_thread.quantile([0.5, 0.9, 0.95, 0.99]).items()},
        "authors_per_thread_percentiles": {str(k): float(v) for k, v in authors_per_thread.quantile([0.5, 0.9, 0.95, 0.99]).items()},
        "duration_hours_percentiles": {str(k): float(v) for k, v in (duration / 3600).quantile([0.5, 0.9, 0.95, 0.99]).items()},
        "author_coverage_pct": 100.0 * (1 - len(missing) / len(authors_in_corpus)),
        "edge_type_counts": {str(k): int(v) for k, v in temporal_edges["edge_type"].value_counts().items()},
        "n_coparticipation_edges": int(len(coparticipation_edges)),
        "threads_with_only_coparticipation": int(len(only_coparticipation)),
        "threads_with_neither_relation": int(len(neither)),
        "chronological_vs_parent_order_match_pct": float(pct),
    },
}

hypergraph_contract_path = os.path.join(ENGINEERED_DIR, "hypergraph_contract.json")
with open(hypergraph_contract_path, "w", encoding="utf-8") as f:
    json.dump(hypergraph_contract, f, indent=2, ensure_ascii=False, default=str)

print("Saved:", hypergraph_contract_path)
print(json.dumps(hypergraph_contract["stats"], indent=2, ensure_ascii=False))


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/engineered/hypergraph_contract.json
{
  "n_hyperedges": 12932,
  "docs_per_thread_percentiles": {
    "0.5": 5.0,
    "0.9": 44.0,
    "0.95": 70.0,
    "0.99": 157.6900000000005
  },
  "authors_per_thread_percentiles": {
    "0.5": 3.0,
    "0.9": 24.0,
    "0.95": 38.0,
    "0.99": 79.69000000000051
  },
  "duration_hours_percentiles": {
    "0.5": 4.379166666666666,
    "0.9": 184.40738888888885,
    "0.95": 665.3292222222195,
    "0.99": 2862.033327777779
  },
  "author_coverage_pct": 100.0,
  "edge_type_counts": {
    "reply_to_comment": 110720,
    "reply_to_post": 83774
  },
  "n_coparticipation_edges": 1891957,
  "threads_with_only_coparticipation": 313,
  "threads_with_neither_relation": 3732,
  "chronological_vs_parent_order_match_pct": 20.63798204602599
}
